In [1]:
import pickle
import numpy as np


class ImageObject:
    def __init__(self, data, label):
        self.data = data
        self.label = label
        self.red = data[:, :, 0]
        self.green = data[:, :, 1]
        self.blue = data[:, :, 2]
        self.grayscale = np.dot(data[..., :3], [0.299, 0.587, 0.114])

    def __repr__(self):
        return (
            f"<ImageObject label={self.label} grayscale_shape={self.grayscale.shape}>"
        )


def load_cifar10():
    try:
        with open("../dataset_cache.pkl", "rb") as f:
            train_data, test_data = pickle.load(f)
        print("Success!")
        print(train_data[0])
        return train_data, test_data

    except FileNotFoundError:
        print("Cache file not found! Please run the setup script first.")
        return [], []

In [22]:
class NN_Classifier:
    def __init__(self, train_data) -> None:
        self.data = np.array([i.data.flatten() for i in train_data]).astype(np.int16)
        self.reds = np.array([i.red.flatten() for i in train_data])
        self.greens = np.array([i.green.flatten() for i in train_data])
        self.blues = np.array([i.blue.flatten() for i in train_data])
        self.grays = np.array([i.grayscale.flatten() for i in train_data])
        self.labels = np.array([i.label for i in train_data])

    def __repr__(self) -> str:
        return f"Data: {len(self.data)}, Labels: {len(self.labels)}\nIndex 0: {self.data[0]}, label: {self.labels[0]}"

    def test(self, test_data, color, metric):
        # Basic grayscale tester with manhattan distance
        results = []
        test_grays = np.array([i.grayscale.flatten() for i in test_data])
        test_img_data = np.array([i.data.flatten() for i in test_data]).astype(np.int16)
        test_labels = [i.label for i in test_data]
        print(
            f"Testing {len(test_data)} images against {len(self.grays)} training samples..."
        )
        gray_count, data_count = 0, 0
        for i in range(len(test_grays)):
            gray_diffs = self.grays - test_grays[i]
            img_diffs = self.data - test_img_data[i]
            gray_abs_diffs = np.abs(gray_diffs)
            img_abs_diffs = np.abs(img_diffs)
            gray_distances = np.sum(gray_abs_diffs, axis=1)
            img_distances = np.sum(img_abs_diffs, axis = 1)
            gray_min_index = np.argmin(gray_distances)
            img_min_index = np.argmin(img_distances)
            gray_predicted_label = self.labels[gray_min_index]
            img_predicted_label = self.labels[img_min_index]
            results.append((gray_predicted_label, img_predicted_label))
            acc_label = test_labels[i]
            if gray_predicted_label == acc_label:
                gray_count += 1
            if img_predicted_label == acc_label:
                data_count += 1
            gray_acc = (gray_count/(i+1))*100
            data_acc = (data_count/(i+1))*100
            if i % 100 == 0:
                print(
                    f"Processed {i}/{len(test_grays)} \n| Pred: {gray_predicted_label}, Accuracy: {gray_acc:2f}% \n| Pred: {img_predicted_label}, Accuracy: {data_acc:2f}% \n| Acctual {test_labels[i]}"
                )
        return results

In [3]:
train_data = []
test_data = []
train_data, test_data = load_cifar10()

print(f"Train size: {len(train_data)}")
print(f"Test size: {len(test_data)}")
xy_train = train_data[:40000]
xy_test = train_data[40000:]

Success!
<ImageObject label=frog grayscale_shape=(32, 32)>
Train size: 50000
Test size: 300000


In [ ]:
classifier = NN_Classifier(xy_train)
print(classifier)
results = classifier.test(xy_test, None, None)
print(results)

Data: 40000, Labels: 40000
Index 0: [ 59  62  63 ... 123  92  72], label: frog
Testing 10000 images against 40000 training samples...
Processed 0/10000 
| Pred: deer, Accuracy: 0.000000% 
| Pred: cat, Accuracy: 0.000000% 
| Acctual automobile
Processed 100/10000 
| Pred: bird, Accuracy: 34.653465% 
| Pred: bird, Accuracy: 41.584158% 
| Acctual bird
